# MCBO actions → SSP vectors → vsa-tree-diffusion training inputs

This notebook builds a dataset for the [vsa-tree-diffusion](../vsa-tree-diffusion-main/README.md) factorizer from an MCBO benchmark task:

1. **Sample actions** with MCBO's own sampler, `search_space.sample(n)`.
2. **Encode** each action with `SSPMCBOAgent.encode` (the `ssp-mcbo` agent used by `experiments/run_agent_mcbo.py`).
3. **Write the atom codebook and per-sample syntax trees** in the form the tree-diffusion trainer accepts.

### What vsa-tree-diffusion takes as training input

`vsa_tree_diffusion.train(...)` does **not** consume a table of vectors. It takes a fixed codebook of `K` unitary atoms
(`init_atoms={name: vector}` or `vsatd train --atoms atoms.npz`) and then generates its own stream of random trees over those
atoms. So the training input we must produce is:

* `atoms.npz` – the agent's slot / filler / axis vectors, keyed by the grammar's atom names (`a, b, c, …`, skipping `s`, then `x25, x26, …`).
* `dataset.npz` – the sampled actions, their agent encodings `V_agent`, and the grammar expression (tree) for each one.
  These are the factorization targets you evaluate a trained model on with `Factorizer.factorize(V)`, and the expressions
  can be used as a structured pool (`training.py` `structured_pool` style) if you want to bias training toward this task's tree shape.

### How the agent's encoding maps onto the grammar

`SSPMCBOAgent.encode` computes

```
phi = S_cont ⊛ phi_cont(x)  +  S_disc ⊛ phi_disc(x)
    + S_nom ⊛ Σ_i (NS_i ⊛ NF_i[x_i])  +  S_ord ⊛ Σ_i (OS_i ⊛ OF_i[x_i])
    + w · (phi_cont ⊛ phi_disc ⊛ NF_0[x_0] ⊛ … )
```

The grammar only has `Bind` (circular convolution), `Bundle` (`(a+b)/√2`), `Sub`, `Inv` over **fixed** atoms, so three adaptations are made:

* **Continuous / integer variables are discretised.** `phi_cont(x) = ⊛_j phi_j(x_j)` exactly (the SSP of a vector is the bind of its per-axis SSPs),
  so each variable gets `N_LEVELS` grid values and each grid value's per-axis SSP becomes an atom. Sampled actions are snapped to the grid
  before encoding; both the raw and snapped actions are saved.
* **`Bundle` normalises by `1/√2` while the agent uses a plain sum.** Bundles are built as balanced binary trees so terms get (near) equal
  weight; the cosine between the agent vector and the grammar-compiled vector is reported and both vectors are saved.
* **The conjunctive term has a scalar weight `w`** which the grammar cannot express. The default here is `CONJUNCTIVE_W = 0`; with `w > 0`
  the term is included un-weighted.

Nothing here trains or runs the tree-diffusion model.

In [ ]:
import os
import sys
import json
import string
from collections import OrderedDict

import numpy as np
import pandas as pd
import torch

# use this repo's ssp_bayes_opt (the mcbo env also has an older copy installed in site-packages)
sys.path.insert(0, os.path.abspath('..'))
from ssp_bayes_opt.agents import SSPMCBOAgent
from mcbo.utils.experiment_utils import get_task_from_id   # MCBO: https://github.com/huawei-noah/HEBO/tree/master/MCBO

# ---------------- parameters ----------------
TASK_ID = 'pest'            # any id accepted by get_task_from_id, e.g. 'pest', 'ackley-53', 'xgboost_opt', 'svm_opt'
N_SAMPLES = 2000            # actions drawn with search_space.sample
N_INIT = 10                 # initial points used to construct the agent (its length-scale fit needs some data)
SSP_DIM = 151               # requested SSP dimension (HexagonalSSPSpace may adjust it; the actual value becomes vec_dim)
N_LEVELS = 16               # grid levels per continuous variable (and per integer variable with a wide range)
CONJUNCTIVE_W = 0.0         # agent's conjunctive weight; 0 keeps the tree an exact description of the encoding
SEED = 0
EVALUATE_OBJECTIVE = False  # also evaluate the task on every sampled action and store it (can be slow for some tasks)
OUT_DIR = os.path.join('data', 'vsa_tree_diffusion', TASK_ID)

np.random.seed(SEED)
torch.manual_seed(SEED)
os.makedirs(OUT_DIR, exist_ok=True)

## 1. Load the MCBO task

The wrapper mirrors `MCBO_task` in `experiments/run_agent_mcbo.py`: MCBO samples in the original units (a `DataFrame`), while the
agent works on the *transformed* numeric representation (continuous → `[0, 1]`, nominal → category index, integers left as is).

In [ ]:
class MCBOTask:
    '''MCBO search-space sampling <-> the numeric arrays SSPMCBOAgent consumes.'''

    def __init__(self, task_id):
        self.task = get_task_from_id(task_id)
        self.search_space = self.task.get_search_space()
        self.param_names = list(self.search_space.params.keys())
        self.disc_names = list(self.search_space.disc_names)
        n = len(self.param_names)
        self.bounds = np.zeros((n, 2))
        self.bounds[:, 0] = [self.search_space.params[p].transfo_lb for p in self.param_names]
        self.bounds[:, 1] = [self.search_space.params[p].transfo_ub for p in self.param_names]

    def sample_df(self, n):
        '''MCBO's built-in random sampler, in original units.'''
        return self.search_space.sample(n)

    def to_array(self, df):
        '''Original-unit DataFrame -> (n, n_params) float array in the agent's representation.'''
        df = df.copy()
        for p in self.param_names:
            if p not in self.disc_names:
                df[p] = self.search_space.params[p].transform(df[p]).cpu().numpy()
        return np.array(df[self.param_names]).astype(float)

    def to_df(self, x):
        '''Inverse of to_array. Copies x: pandas may otherwise write the inverse-transformed columns back into it.'''
        df = pd.DataFrame(np.array(np.atleast_2d(x), dtype=float, copy=True), columns=self.param_names)
        for p in self.param_names:
            if p not in self.disc_names:
                df[p] = self.search_space.params[p].inverse_transform(torch.tensor(df[p].values))
            else:
                df[p] = df[p].astype(int)
        return df

    def evaluate(self, x):
        '''Raw task value (MCBO tasks are minimisation problems).'''
        return np.asarray(self.task(self.to_df(x)), dtype=float).reshape(-1, 1)


target = MCBOTask(TASK_ID)
ss = target.search_space
print(f'task {TASK_ID}: {len(target.param_names)} params -> '
      f'{len(ss.cont_names)} continuous, {len(ss.disc_names)} integer, '
      f'{len(ss.nominal_names)} nominal, {len(ss.ordinal_names)} ordinal, {len(ss.perm_names)} permutation')
assert len(ss.perm_names) == 0, 'SSPMCBOAgent does not support permutation variables'

## 2. Build the `ssp-mcbo` agent

The agent is constructed exactly as `BayesianOptimization.initialize_agent(agent_type='ssp-mcbo')` does, from a handful of initial
points (it fits its length scales on them). Only `agent.encode` and the agent's SSP spaces are used afterwards.

In [ ]:
init_df = target.sample_df(N_INIT)
init_xs = target.to_array(init_df)
init_ys = -target.evaluate(init_xs)   # agent maximises; MCBO tasks minimise

agent = SSPMCBOAgent(init_xs, init_ys, search_space=ss,
                     ssp_dim=SSP_DIM, conjunctive_w=CONJUNCTIVE_W, seed=SEED,
                     decoder_method='direct-optim')
D = agent.ssp_dim
print(f'vector dim (vec_dim for tree diffusion): {D}')
print('length scales (continuous, then integer):', np.ravel(agent.length_scales))

## 3. Atom codebook

Every fixed vector the encoding uses becomes an atom:

| role | atoms |
|---|---|
| type slots | `S_cont`, `S_disc`, `S_nom`, `S_ord` (only those in use) |
| continuous axis `j`, level `k` | per-axis SSP of grid value `k` of variable `j` |
| integer axis `j`, value `k` | per-axis SSP of integer `k` |
| nominal variable `i` | its slot `NS_i` and one filler per category |
| ordinal variable `i` | its slot `OS_i` and one filler per category |

Atoms are named the way `vsa_tree_diffusion.vsa._make_atom_names` names them, so `atoms.npz` lines up with `--num_atoms K`.

In [ ]:
def make_atom_names(num_atoms):
    '''Same naming as vsa_tree_diffusion.vsa._make_atom_names: a..z without "s", then x25, x26, ...'''
    names = [c for c in string.ascii_lowercase if c != 's'][:num_atoms]
    while len(names) < num_atoms:
        names.append(f'x{len(names)}')
    return names


def axis_ssp(space, axis, value):
    '''Per-axis SSP phi_axis(value) so that space.encode(x) == bind_j phi_j(x_j).'''
    ls = np.ravel(space.length_scale)[axis]
    return np.fft.ifft(np.exp(1.j * space.phase_matrix[:, axis] * value / ls)).real


atoms = OrderedDict()     # semantic label -> (D,) vector
atom_info = OrderedDict() # semantic label -> metadata
levels = OrderedDict()    # numeric variable -> grid values (agent representation)
col = {p: i for i, p in enumerate(target.param_names)}

slot_vectors = agent.spaces['slots'].vectors
for k, (lab, n_used) in enumerate([('slot_cont', agent.n_cont), ('slot_disc', agent.n_disc),
                                   ('slot_nominal', agent.n_nomial), ('slot_ordinal', agent.n_ordinal)]):
    if n_used > 0:
        atoms[lab] = slot_vectors[k]
        atom_info[lab] = dict(role='type_slot', param=None, value=None)

if agent.n_cont > 0:
    sp = agent.spaces['cont']
    for j, p in enumerate(ss.cont_names):
        lb, ub = agent.domain_bounds[col[p]]
        levels[p] = np.linspace(lb, ub, N_LEVELS)
        for k, v in enumerate(levels[p]):
            atoms[f'cont:{p}:{k}'] = axis_ssp(sp, j, v)
            atom_info[f'cont:{p}:{k}'] = dict(role='cont_level', param=p, value=float(v))

if agent.n_disc > 0:
    sp = agent.spaces['disc']
    for j, p in enumerate(ss.disc_names):
        lb, ub = int(ss.params[p].lb), int(ss.params[p].ub)
        vals = np.arange(lb, ub + 1)
        if len(vals) > N_LEVELS:
            vals = np.unique(np.round(np.linspace(lb, ub, N_LEVELS)).astype(int))
        levels[p] = vals.astype(float)
        for k, v in enumerate(levels[p]):
            atoms[f'disc:{p}:{k}'] = axis_ssp(sp, j, v)
            atom_info[f'disc:{p}:{k}'] = dict(role='disc_level', param=p, value=int(v))

for kind, names_attr in [('nom', 'nominal'), ('ord', 'ordinal')]:
    names = getattr(ss, f'{names_attr}_names')
    if not names:
        continue
    slots = agent.spaces[f'{names_attr}_slots']
    fillers = agent.spaces[f'{names_attr}_filler']
    for i, p in enumerate(names):
        atoms[f'{kind}slot:{p}'] = slots.vectors[i]
        atom_info[f'{kind}slot:{p}'] = dict(role=f'{names_attr}_slot', param=p, value=None)
        cats = list(getattr(ss.params[p], 'categories', range(ss.params[p].num_uniqs)))
        for c in range(ss.params[p].num_uniqs):
            atoms[f'{kind}:{p}:{c}'] = fillers[p].vectors[c]
            atom_info[f'{kind}:{p}:{c}'] = dict(role=f'{names_attr}_filler', param=p, value=str(cats[c]))

labels = list(atoms.keys())
K = len(labels)
grammar_names = make_atom_names(K)
label_to_name = dict(zip(labels, grammar_names))
atom_matrix = np.stack([atoms[l] for l in labels])

fourier_mag = np.abs(np.fft.fft(atom_matrix, axis=-1))
print(f'{K} atoms of dim {D}; max |1 - Fourier magnitude| = {np.abs(1 - fourier_mag).max():.2e} (unitary check)')
roles = pd.Series([atom_info[l]['role'] for l in labels]).value_counts()
print(roles.to_string())

## 4. Sample actions with MCBO and encode them with the agent

Numeric variables are snapped to their grid so that the saved tree describes the saved vector exactly.

In [ ]:
df_raw = target.sample_df(N_SAMPLES)       # MCBO built-in sampler
x_raw = target.to_array(df_raw)
x = x_raw.copy()

numeric_names = list(ss.cont_names) + list(ss.disc_names)
level_idx = np.zeros((N_SAMPLES, len(numeric_names)), dtype=int)
for j, p in enumerate(numeric_names):
    idx = np.abs(x_raw[:, [col[p]]] - levels[p][None, :]).argmin(axis=1)
    level_idx[:, j] = idx
    x[:, col[p]] = levels[p][idx]

V_agent = agent.encode(x)
print('V_agent', V_agent.shape)
if numeric_names:
    snap_err = np.abs(x - x_raw)[:, [col[p] for p in numeric_names]]
    print('max snapping error per numeric variable:', dict(zip(numeric_names, snap_err.max(axis=0).round(4))))

y_task = target.evaluate(x) if EVALUATE_OBJECTIVE else None

## 5. Syntax tree for every sample, and a check that it compiles to the agent's vector

`compile_expr` re-implements the grammar's semantics in numpy (`Bind` = circular convolution, `Bundle` = `(a+b)/√2`), matching
`vsa_tree_diffusion/vsa.py`. The final optional cell repeats the check with the real package if it is importable.

In [ ]:
def bind_chain(names):
    expr = names[0]
    for n in names[1:]:
        expr = f'(Bind {expr} {n})'
    return expr


def balanced_bundle(terms):
    '''Balanced Bundle tree so every term gets (near) equal weight -- same helper as vsa-tree-diffusion/experiments/common.py.'''
    terms = list(terms)
    while len(terms) > 1:
        merged = [f'(Bundle {terms[i]} {terms[i + 1]})' for i in range(0, len(terms) - 1, 2)]
        if len(terms) % 2:
            merged.append(terms[-1])
        terms = merged
    return terms[0]


def expression_for(i):
    terms, conj = [], []
    if agent.n_cont > 0:
        block = bind_chain([label_to_name[f'cont:{p}:{level_idx[i, numeric_names.index(p)]}'] for p in ss.cont_names])
        conj.append(block)
        terms.append(f'(Bind {label_to_name["slot_cont"]} {block})')
    if agent.n_disc > 0:
        block = bind_chain([label_to_name[f'disc:{p}:{level_idx[i, numeric_names.index(p)]}'] for p in ss.disc_names])
        conj.append(block)
        terms.append(f'(Bind {label_to_name["slot_disc"]} {block})')
    for kind, names_attr in [('nom', 'nominal'), ('ord', 'ordinal')]:
        names = getattr(ss, f'{names_attr}_names')
        if not names:
            continue
        pairs = []
        for p in names:
            filler = label_to_name[f'{kind}:{p}:{int(round(x[i, col[p]]))}']
            conj.append(filler)
            pairs.append(f'(Bind {label_to_name[f"{kind}slot:{p}"]} {filler})')
        terms.append(f'(Bind {label_to_name[f"slot_{names_attr}"]} {balanced_bundle(pairs)})')
    if CONJUNCTIVE_W > 0:
        terms.append(bind_chain(conj))
    return balanced_bundle(terms)


def compile_expr(expr, vecs):
    '''Evaluate a grammar expression with the same operator semantics as vsa_tree_diffusion.vsa.'''
    toks = expr.replace('(', ' ( ').replace(')', ' ) ').split()
    pos = 0

    def bind(a, b):
        return np.fft.ifft(np.fft.fft(a) * np.fft.fft(b)).real

    def parse():
        nonlocal pos
        t = toks[pos]; pos += 1
        if t != '(':
            return vecs[t]
        op = toks[pos]; pos += 1
        args = []
        while toks[pos] != ')':
            args.append(parse())
        pos += 1
        if op == 'Bind':
            return bind(args[0], args[1])
        if op == 'Bundle':
            return (args[0] + args[1]) / np.sqrt(2)
        if op == 'Sub':
            return (args[0] - args[1]) / np.sqrt(2)
        if op == 'Inv':
            return np.concatenate([args[0][:1], np.flip(args[0][1:])])
        raise ValueError(op)

    return parse()


def cosine(a, b):
    return np.sum(a * b, axis=-1) / (np.linalg.norm(a, axis=-1) * np.linalg.norm(b, axis=-1) + 1e-12)


name_to_vec = dict(zip(grammar_names, atom_matrix))
expressions = np.array([expression_for(i) for i in range(N_SAMPLES)])
V_tree = np.stack([compile_expr(e, name_to_vec) for e in expressions])
cos_agent_tree = cosine(V_agent, V_tree)
n_primitives = np.array([sum(t not in ('(', ')', 'Bind', 'Bundle', 'Sub', 'Inv')
                             for t in e.replace('(', ' ( ').replace(')', ' ) ').split()) for e in expressions])

print('example expression:\n ', expressions[0])
print(f'leaves per expression: {n_primitives.min()}..{n_primitives.max()}, '
      f'expression length (chars): {max(len(e) for e in expressions)}')
print(f'cos(V_agent, V_tree): min {cos_agent_tree.min():.4f}  mean {cos_agent_tree.mean():.4f}  '
      '(1.0 only if the encoding has a single additive block; otherwise Bundle normalisation re-weights the blocks)')

# the per-axis factorisation of the SSP spaces is exact:
if agent.n_cont > 0:
    phi = agent.spaces['cont'].encode(x[:, agent.cont_idxs])
    chain = np.stack([compile_expr(bind_chain([label_to_name[f'cont:{p}:{level_idx[i, numeric_names.index(p)]}']
                                               for p in ss.cont_names]), name_to_vec) for i in range(min(50, N_SAMPLES))])
    assert np.allclose(phi[:len(chain)], chain, atol=1e-8), 'per-axis factorisation of the continuous SSP failed'
    print('continuous block: bind of per-axis atoms == spaces["cont"].encode  (ok)')

## 6. Write the dataset

```
data/vsa_tree_diffusion/<task>/
  atoms.npz            K atoms keyed by grammar name  -> vsatd train --atoms atoms.npz --num_atoms K --vec_dim D
  atoms_meta.json      grammar name -> {label, role, param, value}, plus the encoding config
  dataset.npz          x_raw, x, V_agent, V_tree, cos_agent_tree, expressions, n_primitives, level_idx, (y_task)
  samples_raw.csv      MCBO samples in original units, as drawn
  samples_snapped.csv  the snapped actions that were encoded, in original units
  dataset_meta.json    sizes and the train() keyword arguments this dataset implies
```

In [ ]:
np.savez(os.path.join(OUT_DIR, 'atoms.npz'), **{n: v for n, v in zip(grammar_names, atom_matrix)})

with open(os.path.join(OUT_DIR, 'atoms_meta.json'), 'w') as f:
    json.dump(dict(
        task_id=TASK_ID, vec_dim=int(D), num_atoms=int(K), ssp_dim_requested=SSP_DIM, seed=SEED,
        conjunctive_w=CONJUNCTIVE_W, n_levels=N_LEVELS,
        length_scales=np.ravel(agent.length_scales).tolist(),
        param_names=target.param_names,
        levels={p: v.tolist() for p, v in levels.items()},
        atoms={n: dict(label=l, **atom_info[l]) for n, l in zip(grammar_names, labels)},
    ), f, indent=1)

dataset = dict(x_raw=x_raw, x=x, V_agent=V_agent, V_tree=V_tree, cos_agent_tree=cos_agent_tree,
               expressions=expressions, n_primitives=n_primitives, level_idx=level_idx,
               level_idx_columns=np.array(numeric_names), param_names=np.array(target.param_names),
               init_xs=init_xs, init_ys=init_ys)
if y_task is not None:
    dataset['y_task'] = y_task
np.savez(os.path.join(OUT_DIR, 'dataset.npz'), **dataset)

df_raw.to_csv(os.path.join(OUT_DIR, 'samples_raw.csv'), index=False)
target.to_df(x).to_csv(os.path.join(OUT_DIR, 'samples_snapped.csv'), index=False)

train_kwargs = dict(num_atoms=int(K), vec_dim=int(D), no_inv=True, with_sub=False,
                    max_primitives=int(n_primitives.max()),
                    max_sequence_length=int(max(256, 2 * max(len(e) for e in expressions))))
with open(os.path.join(OUT_DIR, 'dataset_meta.json'), 'w') as f:
    json.dump(dict(task_id=TASK_ID, n_samples=int(N_SAMPLES), vec_dim=int(D), num_atoms=int(K),
                   files=['atoms.npz', 'atoms_meta.json', 'dataset.npz', 'samples_raw.csv', 'samples_snapped.csv'],
                   train_kwargs=train_kwargs), f, indent=1)

print('wrote', sorted(os.listdir(OUT_DIR)))
print('implied vsa_tree_diffusion.train kwargs:', train_kwargs)

## 7. Hand-off to vsa-tree-diffusion (not run here)

```python
import numpy as np
from vsa_tree_diffusion import train, Factorizer

atoms = dict(np.load('data/vsa_tree_diffusion/<task>/atoms.npz'))
meta = json.load(open('data/vsa_tree_diffusion/<task>/dataset_meta.json'))
ckpt_dir = train(init_atoms=atoms, **meta['train_kwargs'])          # or: vsatd train --atoms atoms.npz --num_atoms K --vec_dim D --no_inv --no_sub

data = np.load('data/vsa_tree_diffusion/<task>/dataset.npz')
fz = Factorizer(f'{ckpt_dir}/vsa_step_50000.pt')
expr, cos = fz.factorize(data['V_agent'][0])                        # recover the action's tree from its SSP
```

Notes
* `max_primitives` in the trainer counts leaves per tree; MCBO tasks with many categorical variables give large trees
  (see the leaf count printed above), so the curriculum / sequence-length settings should be raised accordingly.
* `V_tree` is what the trainer's compiler would produce for the saved expression; `V_agent` is the vector the BO agent actually uses.
  They differ only by the per-block `1/√2` Bundle weights (and by the conjunctive weight if `CONJUNCTIVE_W > 0`).

In [ ]:
# Optional: confirm with the real package that each saved expression parses under its grammar and compiles to V_tree.
sys.path.insert(0, os.path.join('..', 'vsa-tree-diffusion-main'))
try:
    from vsa_tree_diffusion.vsa import VSA
except ImportError as e:
    print('vsa_tree_diffusion not importable in this environment, skipping check:', e)
else:
    env = VSA(dim=D, seed=SEED, num_atoms=K, init_atoms=dict(np.load(os.path.join(OUT_DIR, 'atoms.npz'))),
              no_inv=True, with_sub=False)
    assert list(env._compiler.atoms.keys()) == grammar_names
    for i in range(min(20, N_SAMPLES)):
        assert cosine(env.compile(str(expressions[i])), V_tree[i]) > 1 - 1e-6
    print('vsa_tree_diffusion parses and compiles the saved expressions to V_tree  (ok)')